# Radial Velocity Analysis of 55 Cancri by Joshua Williams, and Tymari Austin

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

home = os.path.expanduser("~")
file_path = os.path.join(home, "Downloads", "55_Cnc_rv_data.txt")

# Load the data using the path
# Note: We skip 26 rows to bypass the metadata 
data = np.loadtxt(file_path, skiprows=26)

print(f"Successfully loaded {len(data)} points from: {file_path}")
t = data[:, 0]
rv = data[:, 1]
rv_err = data[:, 2]

plt.figure(figsize=(10, 6))
plt.errorbar(t, rv, yerr=rv_err, fmt='o', color='black', ecolor='lightgray', capsize=2, label='Observed RV')
plt.xlabel('Time (Days)')
plt.ylabel('Radial Velocity (m/s)')
plt.title('Radial Velocity of 55 Cnc')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def rv_model(t, K, P, phi, offset):
    return K * np.sin(2 * np.pi * t / P + phi) + offset

# Initial guesses are crucial for curve_fit
# Estimate P by looking at the plot; estimate K by the max RV value
initial_guesses = [50, 14, 0, 0] 

# Increase maxfev to allow more iterations for convergence
# Add bounds to constrain the parameter space if you have prior knowledge
popt, pcov = curve_fit(rv_model, t, rv, p0=initial_guesses, 
                       sigma=rv_err, 
                       maxfev=10000,  # Increased from default 1000
                       method='trf')  # Try different method (Trust Region Reflective)

K_fit, P_fit, phi_fit, offset_fit = popt

print(f"Best Fit: Amplitude K = {K_fit:.2f} m/s, Period P = {P_fit:.2f} days")

In [ ]:
residuals = rv - rv_model(t, *popt)
std_residuals = np.std(residuals)

plt.hist(residuals, bins=15, edgecolor='black')
plt.title('Histogram of Residuals')
plt.xlabel('Residual (m/s)')
plt.ylabel('Frequency')
plt.show()

print(f"Standard Deviation of Residuals: {std_residuals:.2f} m/s")
print(f"Average Experimental Error: {np.mean(rv_err):.2f} m/s")

In [ ]:
m_star = 0.905 # Solar masses
m_planet = (K_fit / 28.4) * (P_fit / 365)**(1/3) * (m_star)**(2/3)

print(f"Calculated Planet Mass: {m_planet:.3f} Jupiter Masses")